# 诗歌生成

# 数据处理

In [6]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('poems.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [7]:
import tensorflow as tf
import keras

class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        
        # 定义嵌入层
        # batch_input_shape=[None, None] 允许动态的 batch_size 和 seq_len
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64, 
                                                    batch_input_shape=[None, None])
        
        # 定义 RNN 单元和层
        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        # return_sequences=True 确保输出每个时间步的 hidden state，而不是只输出最后一步
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        
        # 全连接层，输出维度为词汇表大小
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        '''
        此处完成建模过程
        inp_ids: 输入序列的索引，shape = [batch_size, seq_len]
        '''
        # 1. 嵌入层：将索引转换为向量
        # shape: [batch_size, seq_len, 64]
        x = self.embed_layer(inp_ids)
        
        # 2. RNN 层：提取序列特征
        # shape: [batch_size, seq_len, 128] (因为 return_sequences=True)
        x = self.rnn_layer(x)
        
        # 3. 全连接层：映射到词汇表维度
        # shape: [batch_size, seq_len, v_sz]
        logits = self.dense(x)
        
        return logits
    
    @tf.function
    def get_next_token(self, x, state):
        '''
        用于推理/生成阶段，一次预测一个 token
        shape(x) = [b_sz,] 
        state: 上一步的 hidden state
        '''
        # 1. 嵌入
        # x 是 [b_sz] -> inp_emb 是 [b_sz, 64]
        # 注意：这里需要加一个维度变成 [b_sz, 1, 64] 才能喂给 RNN cell (取决于具体实现，通常 cell 接受 2D 输入)
        inp_emb = self.embed_layer(x) 
        
        # 2. RNN Cell 单步计算
        # SimpleRNNCell 的 call 输入通常是 (inputs, state)
        # inp_emb 是 [b_sz, 64], state 是 [b_sz, 128]
        h, state = self.rnncell.call(inp_emb, state) 
        
        # 3. 输出层
        # h 是 [b_sz, 128] -> logits 是 [b_sz, v_sz]
        logits = self.dense(h) 
        
        # 4. 获取预测结果
        out = tf.argmax(logits, axis=-1) # shape: [b_sz]
        
        return out, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [8]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [9]:
import tensorflow as tf

# 假设 reduce_avg 是你之前定义好的函数，用于处理变长序列的平均损失
# 如果还没定义，这里补一个简单的实现作为参考
def reduce_avg(losses, seqlen, dim=1):
    # losses shape: [batch, seq_len]
    # seqlen shape: [batch,]
    # 将 seqlen 转换为 float32 并扩展维度以便广播
    seqlen = tf.cast(seqlen, tf.float32)
    seqlen = tf.expand_dims(seqlen, axis=dim)
    # 求和然后除以长度
    return tf.reduce_sum(losses, axis=dim) / seqlen

@tf.function
def compute_loss(logits, labels, seqlen):
    """
    计算损失函数
    logits: [batch, seq_len, vocab_size]
    labels: [batch, seq_len]
    seqlen: [batch,]
    """
    # 计算每个位置交叉熵损失
    # result shape: [batch, seq_len]
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    
    # 对序列维度求平均，得到每个样本的损失 shape: [batch,]
    losses = reduce_avg(losses, seqlen, dim=1)
    
    # 对 batch 维度求平均，得到标量损失
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程
    x: 输入数据
    y: 标签数据
    seqlen: 序列长度
    '''
    # 1. 开启梯度记录上下文
    with tf.GradientTape() as tape:
        # 2. 前向传播：获取预测结果
        logits = model(x, training=True)
        
        # 3. 计算损失值
        loss = compute_loss(logits, y, seqlen)
    
    # 4. 反向传播：计算梯度
    # 计算 loss 相对于 model.trainable_variables 的梯度
    gradients = tape.gradient(loss, model.trainable_variables)
    
    # 5. 应用梯度：更新参数
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    # accuracy = 0.0 # 如果需要计算准确率，可以在这里初始化
    
    for step, (x, y, seqlen) in enumerate(ds):
        # 执行一步训练
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [10]:
optimizer = optimizers.Adam(0.0005)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 32.992943
epoch 1 : loss 13.812464
epoch 2 : loss 8.545245
epoch 3 : loss 8.357782
epoch 4 : loss 7.2934217
epoch 5 : loss 7.310827
epoch 6 : loss 7.5611305
epoch 7 : loss 7.036961
epoch 8 : loss 7.106919
epoch 9 : loss 6.984184


# 生成过程

In [11]:
def gen_sentence():
    state = [tf.random.normal(shape=(1, 128), stddev=0.5), tf.random.normal(shape=(1, 128), stddev=0.5)]
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state)
        collect.append(cur_token.numpy()[0])
    return [id2word[t] for t in collect]
print(''.join(gen_sentence()))

何处在，山中去，何处不知君。eosPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPADPAD
